<!-- DRAFT: placeholder prose; the author will rewrite this. -->
# 06 · Questions that need a graph
**You'll learn:** three ways to put a graph to work when vector search isn't enough: enter the graph from names in the question (M7), retrieve wide and rank narrow with a cross-encoder (M8), and give an answering agent graph tools (M9). Measured on gold v2 (12 chain + 8 list questions, frozen before any method ran on them). Agent answers are read from the cache, so this chapter makes no API calls.

In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, '..'); sys.path.insert(0, '../scripts')
import json
from pathlib import Path
from cookbook.db import get_db
from cookbook.load import load_gold_v2
from cookbook.entities import link_entities
from cookbook import retrieve as R
db = get_db()
gold = {q['id']: q for q in load_gold_v2()}
q = gold['v2c01']
print(q['question'])
print('names found:', link_entities(q['question']))

My loan is through Northwind Auto Finance. After I sell you the car, how soon is the loan paid off, and can I skip my next payment?
names found: ['lenders/northwind_finance']


<!-- DRAFT: placeholder prose; the author will rewrite this. -->
**M7**: the lender's name is an entry point into the graph, and it reaches the e-payoff network article (the path is printed). But a hit reached this way inherits 0.8 x the best vector score, so it lands 10th: the graph found the article, the score rule buried it. M8 separates the two jobs: the graph only has to get it into the pool.

In [2]:
for i, r in enumerate(R.retrieve(db, 'M7', q, k=10), 1):
    print(i, r['key'], round(r['score'], 3), r.get('via') or '')

W1118 20:18:00.448000 35108 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

1 lender_overpayment_refund__2022 0.647 
2 payoff_letter__2026 0.594 
3 buy_title_northwind_finance__2022 0.577 
4 buy_title_outside_lender__2022 0.568 
5 getting_paid__2026 0.566 
6 selling_with_loan__2022 0.564 
7 negative_equity__2026 0.552 
8 how_selling_works__2026 0.523 
9 returns__2026 0.519 
10 epayoff_network__2026 0.517 (question) > northwind_finance > epayoff_network__2026


<!-- DRAFT: placeholder prose; the author will rewrite this. -->
**M8, wide**: one AQL query collects candidates from BM25, vector search, graph neighbours and named entities, each tagged with where it came from.

In [3]:
pool = R.pool(db, 'M8', q, show=True)
for c in pool:
    print(c['key'], c['sources'])

AQL> // wide (D-033): every source is filtered to as_of and state
     LET bm25 = (
       FOR d IN kb_view
         SEARCH ANALYZER(d.title IN TOKENS(@q, "text_en") OR d.text IN TOKENS(@q, "text_en"), "text_en")
             OR ANALYZER(d.codes IN @codes, "identity")
         OPTIONS { waitForSync: true }
         FILTER d.effective_from <= @as_of AND (d.effective_to == null OR d.effective_to >= @as_of)
           AND (@state == null OR LENGTH(d.states) == 0 OR @state IN d.states)
         SORT BM25(d) DESC, d._key ASC
         LIMIT @pool_k
         RETURN d._id
     )
     LET vec = (
       FOR d IN articles
         FILTER d.effective_from <= @as_of AND (d.effective_to == null OR d.effective_to >= @as_of)
           AND (@state == null OR LENGTH(d.states) == 0 OR @state IN d.states)
         LET score = APPROX_NEAR_COSINE(d.embedding, @qv)
         SORT score DESC
         LIMIT @pool_k
         RETURN d._id
     )
     // the graph's part: M4's neighbours of the vector top 5, and

<!-- DRAFT: placeholder prose; the author will rewrite this. -->
**M8, narrow**: a cross-encoder reads (question, article) pairs and ranks the pool. The e-payoff network article, which only the graph and the name brought in, moves up to 4th (gold articles are starred).

In [4]:
for r in R.retrieve(db, 'M8', q, k=5):
    print(r['key'], r['score'], r['sources'], '*' if r['key'] in q['gold_keys'] else '')

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

lender_overpayment_refund__2022 4.7343 ['bm25', 'vector', 'graph'] 
buy_title_northwind_finance__2022 3.9141 ['bm25', 'vector', 'name'] 
buy_title_outside_lender__2022 0.9945 ['bm25', 'vector'] 
epayoff_network__2026 0.3016 ['graph', 'name'] *
keep_paying_until_confirmed__2022 -0.8192 ['bm25', 'graph'] *


<!-- DRAFT: placeholder prose; the author will rewrite this. -->
**M9's tools** are fixed AQL templates. The model chooses which to call; it never writes AQL.

In [5]:
from cookbook.tools import execute
out, _ = execute(db, 'lookup', {'kind': 'form', 'name': 'NW-POA-1'}, q, show=True)
print(out)

AQL> LET r = DOCUMENT(@id)
     LET articles = (
       FOR v IN 1 INBOUND r uses_form, about_lender, about_lessor, applies_in
         FILTER v.effective_from <= @as_of AND (v.effective_to == null OR v.effective_to >= @as_of)
         SORT v._key
         RETURN DISTINCT { key: v._key, title: v.title })
     LET replaced_by = (FOR f IN 1 OUTBOUND r replaced_by RETURN f._key)
     LET replaces = (FOR f IN 1 INBOUND r replaced_by RETURN f._key)
     RETURN MERGE(UNSET(r, "_id", "_rev"), { current_articles: articles, replaced_by_form: replaced_by, replaces_form: replaces })
bind: {'id': 'forms/NW-POA-1', 'as_of': '2026-11-01'}
{"_key": "NW-POA-1", "accepted_until": "2026-06-30", "effective_from": "2022-01-01", "name": "Power of attorney (notarized)", "note": "Copies signed before 2026-02-01 are accepted until 2026-06-30.", "replaced_by": "NW-POA-2", "retired_on": "2026-02-01", "side": "sell", "status": "retired", "replaces_form": [], "replaced_by_form": ["NW-POA-2"], "current_articles": 

In [6]:
out, _ = execute(db, 'list_records', {'kind': 'lender', 'field': 'epayoff_network', 'value': 'false'}, q, show=True)
print([r['name'] for r in json.loads(out)])

AQL> FOR r IN @@coll
       FILTER @field == null OR LOWER(TO_STRING(r[@field])) == LOWER(@value)
       SORT r._key
       RETURN UNSET(r, "_id", "_rev")
bind: {'@coll': 'lenders', 'field': 'epayoff_network', 'value': 'false'}
['Coastal Credit Union', 'Harbor Trust Bank', 'Oakline Finance', 'Redstone Credit', 'Summit Peak Bank', 'Valley Motor Credit']


<!-- DRAFT: placeholder prose; the author will rewrite this. -->
A list question, answered by the agent with graph tools (M9) and with search only (M9s), from the cached run 1.

In [7]:
for m in ['M9s', 'M9']:
    r = json.loads(Path(f'../data/eval/answers_v2/run1/{m}/v2s04.json').read_text())
    print(m, [(t['tool'], t['input']) for t in r['agent_meta']['trace']])
    print('  items:', r['items'])
    print('  answer:', r['answer'][:300], '\n')

M9s [('search_help_center', {'query': 'e-payoff network lenders payoff letter'}), ('search_help_center', {'query': 'lenders we work with list of lenders'})]
  items: []
  answer: I can't list the lenders outside the network because the help center only names the lenders that are in it. I don't have access to any record of other lenders. The network lenders are Bayline Credit Union, Northwind Auto Finance, Meridian Auto Loans, Pinewood Federal Credit Union and Lakeshore Bank, 

M9 [('list_records', {'kind': 'lender'}), ('search_help_center', {'query': 'e-payoff network payoff letter lender'})]
  items: ['Coastal Credit Union', 'Harbor Trust Bank', 'Oakline Finance', 'Redstone Credit', 'Summit Peak Bank', 'Valley Motor Credit']
  answer: Six lenders in our records are not in the e-payoff network: Coastal Credit Union, Harbor Trust Bank, Oakline Finance, Redstone Credit, Summit Peak Bank and Valley Motor Credit. Customers with loans from these lenders need to bring a payoff letter to the 

<!-- DRAFT: placeholder prose; the author will rewrite this. -->
Scores on gold v2: retrieval (`make eval-v2`) and answers (`make answers-v2-report`).

In [8]:
from IPython.display import Markdown
Markdown(Path('../data/eval/scores/v2_latest.md').read_text().split('### Per question')[0])

# Retrieval scores, gold v2 (D-031)

Run 2026-11-18T20:00:52 · commit c3b8a50 · gold sha256 59bba6c5881e · k=10, entries=5, decay=0.8, pool 10+10

### Headline (higher is better, except stale and trap)

| method | recall@5 · all (n=20) | recall@10 · all (n=20) | recall@5 · chain (n=12) | all_gold@5 · chain (n=12) | recall@5 · structural (n=8) | recall@10 · structural (n=8) | mrr@10 · all (n=20) | trap@5 · all (n=20) |
|---|---|---|---|---|---|---|---|---|
| M1 BM25 | 0.81 | 0.85 | 0.82 | 0.67 | 0.79 | 0.80 | 0.66 | 0.10 |
| M2 Vector | 0.68 | 0.82 | 0.79 | 0.67 | 0.50 | 0.71 | 0.59 | 0.10 |
| M3 Vector + filter | 0.69 | 0.82 | 0.82 | 0.67 | 0.50 | 0.77 | 0.65 | 0.10 |
| M4 Graph: structural | 0.72 | 0.78 | 0.86 | 0.75 | 0.50 | 0.62 | 0.63 | 0.10 |
| M5 Graph: semantic | 0.69 | 0.77 | 0.82 | 0.67 | 0.50 | 0.64 | 0.63 | 0.10 |
| M6 Graph: both layers | 0.72 | 0.78 | 0.86 | 0.75 | 0.50 | 0.62 | 0.63 | 0.10 |
| M7 Graph: + names | 0.72 | 0.81 | 0.86 | 0.75 | 0.50 | 0.69 | 0.63 | 0.10 |
| M8 Wide -> narrow | 0.85 | 0.90 | 0.89 | 0.75 | 0.79 | 0.93 | 0.71 | 0.10 |
| M8n Wide -> narrow, no graph | 0.83 | 0.89 | 0.86 | 0.75 | 0.79 | 0.93 | 0.71 | 0.10 |
| CE* diagnostic: cross-encoder over every current article | 0.87 | 0.93 | 0.93 | 0.83 | 0.79 | 0.93 | 0.71 | 0.10 |

CE* is not a method: it reranks every article in effect, which only a KB this small allows. If CE* beats M8, the pool is the limit; if not, the reranker is.


### Recall@5 by question type

| type | n | M1 | M2 | M3 | M4 | M5 | M6 | M7 | M8 | M8n |
|---|---|---|---|---|---|---|---|---|---|---|
| chain | 12 | 0.82 | 0.79 | 0.82 | 0.86 | 0.82 | 0.86 | 0.86 | 0.89 | 0.86 |
| structural | 8 | 0.79 | 0.50 | 0.50 | 0.50 | 0.50 | 0.50 | 0.50 | 0.79 | 0.79 |



In [9]:
Markdown(Path('../data/eval/scores/v2_answers.md').read_text())

# Answer eval, gold v2

Mean over the runs, [lowest–highest run]. Facts: judged by Fable 5.2 by meaning. List items: from the answer's `items`, mapped to keys in code (D-035). KB-conflict article: v2c06/v2c07, where a current article still names a retired form. $ per answer: answer + judge, the cost of a shared RAG answer counted once per method.

| method | runs | facts found (20) | all facts (20) | facts, chain (12) | list items found (8) | list items correct (8) | exact list (8) | cites the KB-conflict article ↓ (2) | says not covered ↓ | tool calls | $ per answer |
|---|---|---|---|---|---|---|---|---|---|---|---|
| M3 Vector + filter | 3 | 0.64 [0.64–0.64] | 0.55 [0.55–0.55] | 0.82 [0.82–0.82] | 0.38 [0.38–0.38] | 0.38 [0.38–0.38] | 0.38 [0.38–0.38] | 1.00 [1.00–1.00] | 0.35 [0.35–0.35] | – | 0.023 [0.023–0.023] |
| M4 Graph: structural | 3 | 0.64 [0.63–0.66] | 0.57 [0.55–0.60] | 0.82 [0.81–0.85] | 0.38 [0.38–0.38] | 0.38 [0.38–0.38] | 0.38 [0.38–0.38] | 1.00 [1.00–1.00] | 0.43 [0.40–0.45] | – | 0.015 [0.015–0.015] |
| M7 Graph: + names | 3 | 0.64 [0.63–0.66] | 0.57 [0.55–0.60] | 0.82 [0.81–0.85] | 0.38 [0.38–0.38] | 0.38 [0.38–0.38] | 0.38 [0.38–0.38] | 1.00 [1.00–1.00] | 0.43 [0.40–0.45] | – | 0.000 [0.000–0.000] |
| M8 Wide -> narrow | 3 | 0.74 [0.72–0.75] | 0.63 [0.60–0.65] | 0.83 [0.81–0.85] | 0.60 [0.60–0.60] | 0.75 [0.75–0.75] | 0.50 [0.50–0.50] | 0.83 [0.50–1.00] | 0.33 [0.30–0.40] | – | 0.023 [0.022–0.023] |
| M8n Wide -> narrow, no graph | 3 | 0.74 [0.72–0.75] | 0.63 [0.60–0.65] | 0.83 [0.81–0.85] | 0.60 [0.60–0.60] | 0.75 [0.75–0.75] | 0.50 [0.50–0.50] | 0.83 [0.50–1.00] | 0.33 [0.30–0.40] | – | 0.001 [0.001–0.001] |
| M9s Agent: search only | 3 | 0.88 [0.88–0.88] | 0.85 [0.85–0.85] | 0.94 [0.94–0.94] | 0.79 [0.79–0.79] | 0.88 [0.88–0.88] | 0.75 [0.75–0.75] | 0.50 [0.50–0.50] | 0.15 [0.10–0.20] | 2.5 [2.5–2.5] | 0.034 [0.030–0.040] |
| M9 Agent: search + graph tools | 3 | 0.98 [0.97–0.98] | 0.95 [0.95–0.95] | 0.96 [0.94–0.97] | 1.00 [1.00–1.00] | 1.00 [1.00–1.00] | 1.00 [1.00–1.00] | 0.50 [0.50–0.50] | 0.03 [0.00–0.05] | 2.8 [2.7–2.8] | 0.031 [0.026–0.038] |

Answers: claude-opus-5-5 (effort medium). RAG: top 5 articles, text only. Agents: at most 8 tool calls, fixed AQL templates (`cookbook/tools.py`). Judge: Fable 5.2 (effort low).

Spend (ledger, from 2026-11-18):
```
  agent_M9               claude-opus-5-5     182 calls  $  1.24
  agent_M9s              claude-opus-5-5     164 calls  $  1.42
  agent_dev              claude-opus-5-5       3 calls  $  0.03
  answers                claude-opus-5-5     327 calls  $  3.51
  answers_v2             claude-opus-5-5     159 calls  $  2.07
  extract_semantic       claude-opus-5-5      68 calls  $  0.85
  judge                  claude-fable-5-2    606 calls  $  5.52
  total $14.63 of $50.00
```
